## shift and combining all the b-band images 


In [1]:
# importing all the relevant libraries 
import numpy as np
import astropy
import astroalign as aa
import photutils
import ccdproc
import astropy.io.fits as fits
import matplotlib.pyplot as plt
from ccdproc import CCDData, combiner
from astropy import units as u
from astropy.time import Time
from matplotlib.colors import LogNorm
from photutils.centroids import centroid_com, centroid_2dg, centroid_sources
from photutils.aperture import CircularAperture
from photutils.aperture import aperture_photometry
from photutils.segmentation import detect_sources, deblend_sources, SourceCatalog
from scipy.ndimage import shift


In [2]:
images = ccdproc.ImageFileCollection(".", glob_include='*_b.fit')  # collect all b-band images

for fn in images.files_filtered():  # loop through and print all b-band files
    print(fn)

b_band = [CCDData.read(fn, unit="adu") for fn in images.files_filtered()]  # load all b-band images into a list

Light_NGC_2547_10.0s_IRCUT_20260331-200329_b.fit
Light_NGC_2547_10.0s_IRCUT_20260331-200351_b.fit
Light_NGC_2547_10.0s_IRCUT_20260331-200403_b.fit
Light_NGC_2547_10.0s_IRCUT_20260331-200415_b.fit
Light_NGC_2547_10.0s_IRCUT_20260331-200425_b.fit
Light_NGC_2547_10.0s_IRCUT_20260331-200437_b.fit
Light_NGC_2547_10.0s_IRCUT_20260331-200500_b.fit
Light_NGC_2547_10.0s_IRCUT_20260331-200512_b.fit
Light_NGC_2547_10.0s_IRCUT_20260331-200524_b.fit
Light_NGC_2547_10.0s_IRCUT_20260331-200534_b.fit
Light_NGC_2547_10.0s_IRCUT_20260331-200546_b.fit
Light_NGC_2547_10.0s_IRCUT_20260331-200616_b.fit
Light_NGC_2547_10.0s_IRCUT_20260331-200627_b.fit
Light_NGC_2547_10.0s_IRCUT_20260331-200639_b.fit
Light_NGC_2547_10.0s_IRCUT_20260331-200651_b.fit
Light_NGC_2547_10.0s_IRCUT_20260331-200702_b.fit
Light_NGC_2547_10.0s_IRCUT_20260331-200748_b.fit
Light_NGC_2547_10.0s_IRCUT_20260331-200759_b.fit
Light_NGC_2547_10.0s_IRCUT_20260331-200811_b.fit
Light_NGC_2547_10.0s_IRCUT_20260331-200823_b.fit
Light_NGC_2547_10.0s

In [4]:
images = ccdproc.ImageFileCollection(".",glob_include='*NGC_2547*_b.fit')   # extracts the images from the dataset and stores them in array  

scim = []                                                                    # This is an empty list - scim
for fn in images.files_filtered():
    scim.append(CCDData.read(fn, unit = "adu"))                              # reads the file and adds it to the list 
    # This code will adds MJD-OBS to the header so it will (later) stop producing error messages
    times = scim[-1].header['DATE-OBS']                                      # Get the time
    t = Time(times, format='isot', scale='utc')                              # Read the time from UTC format
    scim[-1].header['MJD-OBS'] = (float(t.mjd), 'MJD') 

In [5]:
newname=[]                                     # new list 
for fn in images.files_filtered():   # filters by header info, and loops through each matching filename
    newname.extend(["S."+fn])                   # adds an s in front of the file name and adds that new string to the list 
print(newname)


['S.Light_NGC_2547_10.0s_IRCUT_20260331-200329_b.fit', 'S.Light_NGC_2547_10.0s_IRCUT_20260331-200351_b.fit', 'S.Light_NGC_2547_10.0s_IRCUT_20260331-200403_b.fit', 'S.Light_NGC_2547_10.0s_IRCUT_20260331-200415_b.fit', 'S.Light_NGC_2547_10.0s_IRCUT_20260331-200425_b.fit', 'S.Light_NGC_2547_10.0s_IRCUT_20260331-200437_b.fit', 'S.Light_NGC_2547_10.0s_IRCUT_20260331-200500_b.fit', 'S.Light_NGC_2547_10.0s_IRCUT_20260331-200512_b.fit', 'S.Light_NGC_2547_10.0s_IRCUT_20260331-200524_b.fit', 'S.Light_NGC_2547_10.0s_IRCUT_20260331-200534_b.fit', 'S.Light_NGC_2547_10.0s_IRCUT_20260331-200546_b.fit', 'S.Light_NGC_2547_10.0s_IRCUT_20260331-200616_b.fit', 'S.Light_NGC_2547_10.0s_IRCUT_20260331-200627_b.fit', 'S.Light_NGC_2547_10.0s_IRCUT_20260331-200639_b.fit', 'S.Light_NGC_2547_10.0s_IRCUT_20260331-200651_b.fit', 'S.Light_NGC_2547_10.0s_IRCUT_20260331-200702_b.fit', 'S.Light_NGC_2547_10.0s_IRCUT_20260331-200748_b.fit', 'S.Light_NGC_2547_10.0s_IRCUT_20260331-200759_b.fit', 'S.Light_NGC_2547_10.0s_IRC

In [8]:
# Make a copy of the image 
temp=scim[0].copy()
# Subtract the background 
temp=temp-np.ma.median(temp)

starxy = [(618,1046)]   # Star positon 
bs = 23                 # Box size - 39x39 pixel around the star
xc = int(starxy[0][0])   # guesses for the centroid algorithm
yc = int(starxy[0][1])   # guesses for the centroid algorithm

x1, y1 = centroid_sources(temp, xc, yc, box_size=bs, centroid_func=centroid_com)
print('Centroid com:', x1, y1)

# You will need to add some code here 
x2, y2 = centroid_sources(temp, xc, yc, box_size=bs, centriod_func=centroid_2dg)
print('Centriod 2D Gaussian', x2, y2)

Centroid com: [618.02951837] [1044.18545709]
Centriod 2D Gaussian [618.02951837] [1044.18545709]


In [10]:
starxy = []

#proc_NGC_3766_R_00004556.fits
starxy.append((618,1046))        # Initial estimate of star positon
#proc_NGC_3766_R_00004557.fits
starxy.append((618,1046))        # Initial estimate of star positon
#proc_NGC_3766_R_00004558.fits
starxy.append((618,1046))        # Update?
#proc_NGC_3766_R_00004559.fits
starxy.append((618,1046))        # Update?
#proc_NGC_3766_R_00004560.fits
starxy.append((618,1046))        # Update?
#proc_NGC_3766_R_00004641.fits
starxy.append((618,1046))        # Update?
#proc_NGC_3766_R_00004642.fits
starxy.append((618,1046))        # Update?
#proc_NGC_3766_R_00004643.fits
starxy.append((618,1046))        # Update?
#proc_NGC_3766_R_00004644.fits
starxy.append((618,1046))        # Update?
#proc_NGC_3766_R_00004645.fits
starxy.append((618,1046))        # Update?

# Step through the images and produce a list of positions 
newstarxy=[]
for idx, thisimage in enumerate(scim): 
    xc = int(starxy[idx][0])   # x position guess
    yc = int(starxy[idx][1])   # y position guess
    bs = 23                    # Box size - 39x39 pixel square around the star 
    temp=scim[idx].copy()
    temp=temp-np.ma.median(temp)
    # What should go here? What is missing?
    x1,y1= centroid_sources(temp, xc, yc, box_size=bs, centroid_func=centroid_2dg)
    
    newstarxy.append((x1[0], y1[0]))

print(newstarxy)



IndexError: list index out of range